# Archived weather forecasts for Citi Bike stations

We assign archived ECMWF IFS weather forecasts to Citi Bike station locations for our departure-forecasting project. We preserve the forecast origins, horizons, selected runs, fallback flags, and missing values.

We match each station to the nearest archived request coordinate. This is a geographic approximation, not a forecast requested at that station. We store the weather and station mapping separately, then join them through a persistent view to avoid duplicating weather values.


## 1. Prepare and upload the inputs

We use the normalized request-class forecasts from the existing weather archive. This preparation script runs locally where those Parquet files are stored; it converts timestamp storage to microseconds for Spark without changing the original files. PyArrow is required for this local step.

```python
from pathlib import Path
from io import BytesIO
import hashlib
import json
import zipfile
import pyarrow.parquet as pq

# Locating the completed request-class forecasts from the original notebook.
project_root = Path(r"C:\path\to\weather_project")  # Setting this to the weather project folder.
source = project_root / (
    "data/silver/weather_forecasts/open_meteo_single_runs/ecmwf_ifs/"
    "selected_origin_horizon_request_classes/availability_delay=7h"
)
files = sorted(source.glob("year=*/month=*/selected_weather_request_classes_*.parquet"))
assert files, f"No completed class Parquet files found: {source}"
output = project_root / "citibike_weather_input.zip"
manifest = {"format_version": 1, "source_root": str(source), "files": []}
with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        table = pq.read_table(path, partitioning=None)
        buffer = BytesIO()
        pq.write_table(table, buffer, compression="zstd", coerce_timestamps="us",
                       allow_truncated_timestamps=False)
        payload = buffer.getvalue()
        name = "weather_classes/" + path.relative_to(source).as_posix()
        archive.writestr(name, payload)
        manifest["files"].append({"path": name, "size": len(payload),
                                  "sha256": hashlib.sha256(payload).hexdigest()})
    archive.writestr("input_manifest.json", json.dumps(manifest, indent=2))
print("Package:", output)
print("Monthly files:", len(files))
print("Size (MiB):", round(output.stat().st_size / 2**20, 2))
```

After creating the volume in the next cell, we upload `citibike_weather_input.zip` and `station_information.json` to its root. We skip local packaging when the input ZIP is already available. To import the completed dataset instead of rebuilding it, we use Section 15.


In [0]:
# Creating a schema and volume for the weather project files.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.citibike_weather")

spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.citibike_weather.transfer
""")

print("Upload location: /Volumes/workspace/citibike_weather/transfer")

Upload location: /Volumes/workspace/citibike_weather/transfer


## 2. Load the Citi Bike station snapshot

We validate station IDs and coordinates before saving the snapshot. The recorded `fetched_at` is the import time, not the original feed publication time.


In [0]:
from pyspark.sql import functions as F

# Reading the downloaded station information snapshot.
station_file = (
    "/Volumes/workspace/citibike_weather/transfer/"
    "station_information.json"
)

station_feed = spark.read.option("multiLine", "true").json(station_file)

# Expanding the station array into one row per station.
station_records = (
    station_feed
    .select(F.explode("data.stations").alias("station"))
    .select("station.*")
)

print("Station records:", station_records.count())

display(
    station_records.select(
        "station_id", "name", "lat", "lon", "capacity"
    ).limit(10)
)

Station records: 2519


station_id,name,lat,lon,capacity
0687075b-cf2d-4130-a859-fe841aafbd3f,Vernon Blvd & 41 Rd,40.7552433,-73.9485678,27
2206779090602992084,68 Dr & 112 St,40.72704,-73.84128,12
2250952758815752682,Dumont Ave & Barbey St,40.66759,-73.88609,12
c2d549f8-f54c-4a0d-9ecd-6e6f79118279,Cooper Ave & 65 Pl,40.69953,-73.88873,23
66dd26f9-0aca-11e7-82f6-3863bb44ef7c,9 St & 44 Rd,40.74966,-73.9521,34
595cf79f-7783-4373-b3d5-775bcf76033b,Ogden Ave & Merriam Ave,40.839142,-73.925961,19
6fb81fda-f0b5-4242-90f9-88d0214feb8d,Sedgwick Ave & W Tremont Ave,40.85304455868985,-73.91830623149872,19
1917312704757770810,1 Ave & E 38 St,40.746202,-73.971822,39
ef6216aa-d3c8-46bb-8c6b-b62f8ba51b16,Van Sinderen Ave & Truxton St,40.6786,-73.90369,23
d979131e-0b9f-4aa2-88c5-46d4ff94abc8,Pacific St & Sackman St,40.67522,-73.9061,19


In [0]:
# Standardizing the fields needed for weather mapping.
stations_clean = station_records.select(
    F.trim(F.col("station_id").cast("string")).alias("station_id"),
    F.col("name").cast("string").alias("name"),
    F.col("lat").cast("double").alias("lat"),
    F.col("lon").cast("double").alias("lon"),
)

# Checking identifiers and geographic coordinates.
station_quality = stations_clean.agg(
    F.count("*").alias("rows"),
    F.countDistinct("station_id").alias("unique_station_ids"),
    F.sum(
        F.when(
            F.col("station_id").isNull() | (F.col("station_id") == ""),
            1,
        ).otherwise(0)
    ).alias("missing_ids"),
    F.sum(
        F.when(
            F.col("lat").isNull()
            | F.col("lon").isNull()
            | F.isnan("lat")
            | F.isnan("lon")
            | ~F.col("lat").between(-90, 90)
            | ~F.col("lon").between(-180, 180),
            1,
        ).otherwise(0)
    ).alias("invalid_coordinates"),
)

display(station_quality)

quality = station_quality.first()

assert quality["missing_ids"] == 0, "Missing station IDs."
assert quality["unique_station_ids"] == quality["rows"], "Duplicate station IDs."
assert quality["invalid_coordinates"] == 0, "Invalid station coordinates."

rows,unique_station_ids,missing_ids,invalid_coordinates
2519,2519,0,0


In [0]:
# Checking which tables were actually created in the project schema.
display(spark.sql("SHOW TABLES IN workspace.citibike_weather"))

database,tableName,isTemporary


In [0]:
station_table = "workspace.citibike_weather.station_information_snapshot"

# Recording the snapshot's import time and source.
station_snapshot = (
    stations_clean
    .withColumn("fetched_at", F.current_timestamp())
    .withColumn("source_file", F.lit(station_file))
    .withColumn("snapshot_timestamp_basis", F.lit("import_time"))
)

# Making the validated records available to SQL.
station_snapshot.createOrReplaceTempView("station_snapshot_to_save")

# Creating the Delta table without replacing an existing table.
spark.sql(f"""
    CREATE TABLE {station_table}
    USING DELTA
    AS SELECT * FROM station_snapshot_to_save
""")

print("Saved table:", station_table)
print("Saved stations:", spark.table(station_table).count())

Saved table: workspace.citibike_weather.station_information_snapshot
Saved stations: 2519


## 3. Configure the notebook

We set the input paths, table names, forecast horizons, and export options here. We use a new `DATASET_LABEL` when changing the input data or station snapshot.


In [0]:
from pathlib import Path
from functools import reduce
import hashlib
import json
import re
import zipfile
from datetime import datetime, timezone
from pyspark.sql import functions as F, Window

# Defining workspace names independently of the team pipeline configuration.
CATALOG = "workspace"
SCHEMA = "citibike_weather"
VOLUME = "transfer"
STATION_SOURCE_TABLE = "workspace.citibike_weather.station_information_snapshot"  # Editing to match my actual table.
DATASET_LABEL = "archive_v1"  # Using a new label when changing station snapshots or inputs.

# Selecting an optional UTC origin interval; None means all uploaded origins.
ORIGIN_START_UTC = None
ORIGIN_END_UTC = None  # Treating the end as exclusive.
STATION_AS_OF_UTC = None  # None means latest snapshot present when this notebook runs.
HORIZONS = [1, 2, 3, 4, 5, 6]
EXPECTED_CLASS_COUNT = 28
ASSUMED_AVAILABILITY_DELAY_HOURS = 7
MAX_MAPPING_DISTANCE_KM = 10.0  # Using a review threshold, not a validated accuracy guarantee.
ALLOW_RELOCATED_STATIONS = False

# Keeping the complete dataset compact by default; enabling expansion only if needed.
MATERIALIZE_STATION_TABLE = False
EXPORT_PORTABLE_DATA = True
EXPORT_EXPANDED_DATA = False
EXPORT_PARTITIONS_PER_MONTH = 4

for value in [CATALOG, SCHEMA, VOLUME, DATASET_LABEL]:
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), value
assert len(STATION_SOURCE_TABLE.split(".")) == 3
assert all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", part)
           for part in STATION_SOURCE_TABLE.split("."))
assert MAX_MAPPING_DISTANCE_KM > 0
assert EXPORT_PARTITIONS_PER_MONTH > 0

spark.conf.set("spark.sql.session.timeZone", "UTC")
NAMESPACE = f"{CATALOG}.{SCHEMA}"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PACKAGE_PATH = Path(VOLUME_ROOT) / "citibike_weather_input.zip"
CLASS_TABLE = f"{NAMESPACE}.{DATASET_LABEL}_weather_classes"
MAPPING_TABLE = f"{NAMESPACE}.{DATASET_LABEL}_station_weather_mapping"
FORECAST_VIEW = f"{NAMESPACE}.{DATASET_LABEL}_station_forecast"
EXPANDED_TABLE = f"{NAMESPACE}.{DATASET_LABEL}_station_forecast_materialized"
EXPORT_ROOT = f"{VOLUME_ROOT}/exports/{DATASET_LABEL}"

print("Station source:", STATION_SOURCE_TABLE)
print("Input package:", PACKAGE_PATH)
print("Saved station view:", FORECAST_VIEW)
print("Origin interval:", ORIGIN_START_UTC, "to", ORIGIN_END_UTC)
print("Expanded physical table:", MATERIALIZE_STATION_TABLE)
print("Spark version:", spark.version)


Station source: workspace.citibike_weather.station_information_snapshot
Input package: /Volumes/workspace/citibike_weather/transfer/citibike_weather_input.zip
Saved station view: workspace.citibike_weather.archive_v1_station_forecast
Origin interval: None to None
Expanded physical table: False
Spark version: 4.2.0


## 4. Create persistent storage

We use a Unity Catalog volume for files and managed Delta tables for the compact dataset.


In [0]:
# Creating storage objects only when they do not already exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NAMESPACE}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {NAMESPACE}.{VOLUME}")
print("Volume ready:", VOLUME_ROOT)
print("I can now upload citibike_weather_input.zip to this volume's root.")


## 5. Verify and extract the input package

We verify file sizes and SHA-256 hashes against the input manifest before extracting the package.


In [0]:
# Verifying the package before placing its contents in the managed volume.
assert PACKAGE_PATH.is_file(), f"Upload the local package first: {PACKAGE_PATH}"
package_sha256 = hashlib.file_digest(PACKAGE_PATH.open("rb"), "sha256").hexdigest()
extract_root = Path(VOLUME_ROOT) / "inputs" / package_sha256
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    assert len(archive.namelist()) == len(set(archive.namelist())), "Duplicate ZIP members."
    manifest = json.loads(archive.read("input_manifest.json"))
    assert manifest["format_version"] == 1
    records = manifest["files"]
    assert records and len(records) == len({r["path"] for r in records})
    assert set(archive.namelist()) == {r["path"] for r in records} | {"input_manifest.json"}
    for record in records:
        relative = Path(record["path"])
        assert not relative.is_absolute() and ".." not in relative.parts
        assert relative.parts[0] == "weather_classes" and relative.suffix == ".parquet"
        payload = archive.read(record["path"])
        assert len(payload) == record["size"]
        assert hashlib.sha256(payload).hexdigest() == record["sha256"]
        destination = extract_root / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(payload)
WEATHER_INPUT_ROOT = str(extract_root / "weather_classes")
print("Verified monthly files:", len(records))
print("Package SHA-256:", package_sha256)
print("Spark input:", WEATHER_INPUT_ROOT)


Verified monthly files: 23
Package SHA-256: bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
Spark input: /Volumes/workspace/citibike_weather/transfer/inputs/bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9/weather_classes


## 6. Read the archived forecasts

We load the selected request-class forecasts while retaining their original model-run and fallback metadata.


In [0]:
# Reading only the completed Parquet files and preserving their source paths.
weather = (spark.read.option("recursiveFileLookup", "true")
           .option("pathGlobFilter", "selected_weather_request_classes_*.parquet")
           .parquet(WEATHER_INPUT_ROOT))
WEATHER_VALUES = ["forecast_temperature_f", "forecast_precipitation_in",
                  "forecast_rain_in", "forecast_snowfall_in", "forecast_wind_speed_mph"]
TIMESTAMPS = ["forecast_origin_utc", "target_hour_utc", "selected_model_run_utc",
              "selected_run_available_utc", "valid_time_utc"]
SCHEDULE_COLUMNS = ["target_hour_utc", "selected_model_run_utc", "selected_run_available_utc",
                    "selected_run_age_hours", "weather_lead_hours", "is_fallback_weather_run",
                    "selected_forecast_hours", "selected_weather_source", "selected_raw_json_relative_path"]
REQUIRED = (TIMESTAMPS + WEATHER_VALUES + ["weather_request_class", "forecast_horizon",
            "representative_location_id", "request_latitude", "request_longitude",
            "model_grid_latitude", "model_grid_longitude", "selected_run_age_hours",
            "weather_lead_hours", "is_fallback_weather_run", "selected_forecast_hours",
            "selected_weather_source", "selected_raw_json_relative_path"])
assert not (set(REQUIRED) - set(weather.columns)), sorted(set(REQUIRED) - set(weather.columns))
weather = weather.select(*REQUIRED)
for name in TIMESTAMPS:
    weather = weather.withColumn(name, F.col(name).cast("timestamp"))
for name in WEATHER_VALUES:
    weather = weather.withColumn(name, F.when(F.isnan(F.col(name)), None).otherwise(F.col(name)).cast("float"))
for name in ["weather_request_class", "forecast_horizon", "representative_location_id",
             "selected_run_age_hours", "weather_lead_hours", "selected_forecast_hours"]:
    weather = weather.withColumn(name, F.col(name).cast("int"))
weather = weather.withColumn("is_fallback_weather_run", F.col("is_fallback_weather_run").cast("boolean"))
if ORIGIN_START_UTC is not None:
    weather = weather.filter(F.col("forecast_origin_utc") >= F.to_timestamp(F.lit(ORIGIN_START_UTC)))
if ORIGIN_END_UTC is not None:
    weather = weather.filter(F.col("forecast_origin_utc") < F.to_timestamp(F.lit(ORIGIN_END_UTC)))
weather = (weather.withColumn("origin_year", F.year("forecast_origin_utc"))
           .withColumn("origin_month", F.month("forecast_origin_utc"))
           .withColumn("weather_forecast_has_missing", reduce(lambda a, b: a | b,
                        [F.col(name).isNull() for name in WEATHER_VALUES])))
weather.printSchema()
display(weather.limit(6))


root
 |-- forecast_origin_utc: timestamp (nullable = true)
 |-- target_hour_utc: timestamp (nullable = true)
 |-- selected_model_run_utc: timestamp (nullable = true)
 |-- selected_run_available_utc: timestamp (nullable = true)
 |-- valid_time_utc: timestamp (nullable = true)
 |-- forecast_temperature_f: float (nullable = true)
 |-- forecast_precipitation_in: float (nullable = true)
 |-- forecast_rain_in: float (nullable = true)
 |-- forecast_snowfall_in: float (nullable = true)
 |-- forecast_wind_speed_mph: float (nullable = true)
 |-- weather_request_class: integer (nullable = true)
 |-- forecast_horizon: integer (nullable = true)
 |-- representative_location_id: integer (nullable = true)
 |-- request_latitude: double (nullable = true)
 |-- request_longitude: double (nullable = true)
 |-- model_grid_latitude: double (nullable = true)
 |-- model_grid_longitude: double (nullable = true)
 |-- selected_run_age_hours: integer (nullable = true)
 |-- weather_lead_hours: integer (nullable = t

forecast_origin_utc,target_hour_utc,selected_model_run_utc,selected_run_available_utc,valid_time_utc,forecast_temperature_f,forecast_precipitation_in,forecast_rain_in,forecast_snowfall_in,forecast_wind_speed_mph,weather_request_class,forecast_horizon,representative_location_id,request_latitude,request_longitude,model_grid_latitude,model_grid_longitude,selected_run_age_hours,weather_lead_hours,is_fallback_weather_run,selected_forecast_hours,selected_weather_source,selected_raw_json_relative_path,origin_year,origin_month,weather_forecast_has_missing
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.6,0.0,0.0,0.0,7.3,0,1,44,40.52549110299151,-74.23353546529498,40.52724,-74.237274,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,64.8,0.0,0.0,0.0,6.9,1,1,109,40.55186202658534,-74.15089028772675,40.597538,-74.215,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,64.9,0.0,0.0,0.0,6.9,2,1,5,40.55265878171833,-74.18848459684874,40.597538,-74.215,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.4,0.0,0.0,0.0,7.7,3,1,176,40.56199406192752,-74.12258304615597,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.1,0.0,0.0,0.0,7.7,4,1,14,40.62483367348954,-74.0298925091807,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.4,0.0,0.0,0.0,7.7,5,1,110,40.54577972951617,-74.1283431099553,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false


## 7. Validate forecast timing and coverage

We check the forecast keys, consecutive hourly origins, and complete origin–horizon–class combinations. We retain UTC for time comparisons and joins.

The seven-hour availability delay is an assumption, not verified historical publication timing. The early ECMWF archive includes hindcasts, so this dataset alone does not establish an exact historical operational backtest.


In [0]:
def assert_empty(frame, message):
    assert not frame.limit(1).count(), message

def assert_unique(frame, keys, label):
    assert_empty(frame.groupBy(*keys).count().filter(F.col("count") != 1),
                 f"Duplicate {label}: {keys}")

# Checking required metadata, timestamp arithmetic, and completeness of the schedule.
nonnull = [name for name in REQUIRED if name not in WEATHER_VALUES]
assert_empty(weather.filter(reduce(lambda a, b: a | b,
             [F.col(name).isNull() for name in nonnull])), "Missing required weather metadata.")
assert_unique(weather, ["forecast_origin_utc", "forecast_horizon", "weather_request_class"], "weather keys")
origin_seconds = F.col("forecast_origin_utc").cast("long")
target_seconds = F.col("target_hour_utc").cast("long")
run_seconds = F.col("selected_model_run_utc").cast("long")
available_seconds = F.col("selected_run_available_utc").cast("long")
bad_time = (
    (target_seconds != origin_seconds + F.col("forecast_horizon") * 3600)
    | (target_seconds != run_seconds + F.col("weather_lead_hours") * 3600)
    | (origin_seconds != run_seconds + F.col("selected_run_age_hours") * 3600)
    | (available_seconds != run_seconds + ASSUMED_AVAILABILITY_DELAY_HOURS * 3600)
    | (available_seconds > origin_seconds)
    | (F.col("valid_time_utc") != F.col("target_hour_utc"))
    | (F.col("weather_lead_hours") >= F.col("selected_forecast_hours"))
    | (F.col("weather_lead_hours") < 0)
    | (origin_seconds % 3600 != 0)
    | (~F.col("forecast_horizon").isin(HORIZONS))
    | (F.abs(target_seconds - F.col("target_hour_utc").cast("double")) > 0)
)
assert_empty(weather.filter(bad_time), "Weather timing or horizon contract failed.")
coverage = weather.groupBy("forecast_origin_utc", "forecast_horizon").agg(
    F.countDistinct("weather_request_class").alias("classes"),
    F.countDistinct(F.struct(*SCHEDULE_COLUMNS)).alias("schedules"))
assert_empty(coverage.filter((F.col("classes") != EXPECTED_CLASS_COUNT)
                             | (F.col("schedules") != 1)), "Class coverage or schedule agreement failed.")
assert_empty(coverage.groupBy("forecast_origin_utc").count().filter(F.col("count") != len(HORIZONS)),
             "Missing horizon at an origin.")
summary = weather.agg(F.count("*").alias("rows"),
    F.countDistinct("forecast_origin_utc").alias("origins"),
    F.min("forecast_origin_utc").alias("first_origin"),
    F.max("forecast_origin_utc").alias("last_origin"),
    F.sum(F.col("weather_forecast_has_missing").cast("long")).alias("incomplete_rows")).first()
assert summary.rows > 0, "No forecasts in selected interval."
assert summary.rows == summary.origins * len(HORIZONS) * EXPECTED_CLASS_COUNT
assert int((summary.last_origin - summary.first_origin).total_seconds() / 3600) + 1 == summary.origins, "Missing hourly origins."
assert weather.select("weather_request_class").distinct().count() == EXPECTED_CLASS_COUNT
print(summary.asDict())
display(weather.groupBy("origin_year", "origin_month").agg(
    F.count("*").alias("class_rows"),
    F.sum(F.col("weather_forecast_has_missing").cast("long")).alias("incomplete_rows"),
    F.sum(F.col("is_fallback_weather_run").cast("long")).alias("fallback_class_rows")
).orderBy("origin_year", "origin_month"))


{'rows': 2780568, 'origins': 16551, 'first_origin': datetime.datetime(2024, 8, 10, 7, 0), 'last_origin': datetime.datetime(2026, 6, 30, 21, 0), 'incomplete_rows': 23184}


origin_year,origin_month,class_rows,incomplete_rows,fallback_class_rows
2024,8,87528,0,0
2024,9,120960,0,0
2024,10,124992,0,0
2024,11,120960,0,0
2024,12,124992,0,0
2025,1,124992,0,0
2025,2,112896,0,0
2025,3,124992,0,0
2025,4,120960,0,0
2025,5,124992,0,0


## 8. Select station locations

We select the latest available record for each `station_id`. This static snapshot does not establish historical station operating coverage. A single snapshot also cannot reveal earlier station relocations.

We retain missing weather values and their flags rather than replacing them with zero.


In [0]:
# Selecting station snapshots without hiding invalid latest locations behind older valid records.
assert spark.catalog.tableExists(STATION_SOURCE_TABLE), f"Station table not found: {STATION_SOURCE_TABLE}"
stations_raw = spark.table(STATION_SOURCE_TABLE)
assert {"station_id", "name", "lat", "lon", "fetched_at"} <= set(stations_raw.columns)
stations_raw = stations_raw.select(
    F.trim(F.col("station_id").cast("string")).alias("station_id"),
    F.col("name").cast("string").alias("station_name"),
    F.col("lat").cast("double").alias("station_latitude"),
    F.col("lon").cast("double").alias("station_longitude"),
    F.col("fetched_at").cast("timestamp").alias("station_metadata_fetched_at_utc"))
assert_empty(stations_raw.filter(F.col("station_id").isNull() | (F.col("station_id") == "")
    | F.col("station_metadata_fetched_at_utc").isNull()), "Missing station ID or collection timestamp.")
if STATION_AS_OF_UTC is not None:
    stations_raw = stations_raw.filter(F.col("station_metadata_fetched_at_utc") <= F.to_timestamp(F.lit(STATION_AS_OF_UTC)))
station_cutoff = stations_raw.agg(F.max("station_metadata_fetched_at_utc")).first()[0]
assert station_cutoff is not None, "No station snapshots at this cutoff."
latest_times = stations_raw.groupBy("station_id").agg(
    F.max("station_metadata_fetched_at_utc").alias("station_metadata_fetched_at_utc"))
latest = stations_raw.join(latest_times, ["station_id", "station_metadata_fetched_at_utc"], "inner").distinct()
assert_unique(latest, ["station_id"], "latest station records (conflicting same-time metadata)")
invalid_station = (F.col("station_latitude").isNull() | F.isnan("station_latitude")
    | F.col("station_longitude").isNull() | F.isnan("station_longitude")
    | ~F.col("station_latitude").between(-90, 90)
    | ~F.col("station_longitude").between(-180, 180))
assert_empty(latest.filter(invalid_station), "Invalid coordinates in the latest station snapshot.")
location_versions = stations_raw.groupBy("station_id").agg(F.countDistinct(F.struct(
    F.round("station_latitude", 5), F.round("station_longitude", 5))).alias("station_location_version_count"))
stations = latest.join(location_versions, "station_id")
relocated_count = stations.filter(F.col("station_location_version_count") > 1).count()
print("Selected station cutoff (UTC):", station_cutoff)
print("Stations with changed coordinates in available snapshots:", relocated_count)
assert ALLOW_RELOCATED_STATIONS or relocated_count == 0, "Review relocations before enabling ALLOW_RELOCATED_STATIONS."
station_count = stations.count()
assert station_count > 0
print("Stations to map:", station_count)
display(stations.orderBy("station_id").limit(10))


Selected station cutoff (UTC): 2026-10-18 02:42:08.995720
Stations with changed coordinates in available snapshots: 0
Stations to map: 2519


station_id,station_metadata_fetched_at_utc,station_name,station_latitude,station_longitude,station_location_version_count
00284700-9d22-42ce-8485-113fed9879c1,2026-10-18T02:42:08.995Z,28 Ave & 44 St,40.76408932350688,-73.91065120697021,1
002e6e9f-ced2-47b0-bc7e-ccf38c3b4b0f,2026-10-18T02:42:08.995Z,Prospect Ave & E 151 St,40.814232,-73.903927,1
00967b8f-1a4d-4131-a65a-17fa8ca89e28,2026-10-18T02:42:08.995Z,41 Ave & 67 St,40.74446,-73.89764,1
0096fecb-3a56-4585-9204-aa9554f7bc49,2026-10-18T02:42:08.995Z,E 5 St & Ave A,40.72479007746337,-73.98430064320564,1
00d3e65e-b614-4408-aaa9-939ed96ce645,2026-10-18T02:42:08.995Z,Perry St & Greenwich Ave,40.735918,-74.000939,1
00d94dfc-3695-47e1-aba6-7f319dd2eb08,2026-10-18T02:42:08.995Z,Bergen St & Kingston Ave,40.6756,-73.9415,1
0109f17a-2378-4500-9648-88bdb31ec612,2026-10-18T02:42:08.995Z,Degraw St & 3 Ave,40.67977,-73.9847,1
0138452a-b9f4-4aee-80b0-7fae9a122ffe,2026-10-18T02:42:08.995Z,E 54 St & 1 Ave,40.756265,-73.964179,1
014a931b-8e1c-438a-8a22-f267ddd1e0d2,2026-10-18T02:42:08.995Z,Peck Slip & South St,40.7075188830923,-74.00108054280281,1
016ee47a-4871-4209-91cb-bf3ceeddc991,2026-10-18T02:42:08.995Z,St Felix Ave & 61 St,40.69715,-73.8936,1


## 9. Map stations to weather request locations

We use great-circle distance to find the nearest archived request coordinate for each station. We review mapping distances and stop if any exceed `MAX_MAPPING_DISTANCE_KM`. This threshold is a review rule, not a guarantee of weather accuracy.


In [0]:
# Building a small, stable location lookup before joining it to the time series.
# Standardizing coordinate precision for the mapping lookup only.
representatives = (
    weather.select(
        "weather_request_class",
        "representative_location_id",
        F.round("request_latitude", 7).alias("request_latitude"),
        F.round("request_longitude", 7).alias("request_longitude"),
    )
    .distinct()
)
assert_unique(representatives, ["weather_request_class"], "representative coordinates")
assert_empty(representatives.filter(F.col("request_latitude").isNull()
    | F.isnan("request_latitude") | ~F.col("request_latitude").between(-90, 90)
    | F.col("request_longitude").isNull() | F.isnan("request_longitude")
    | ~F.col("request_longitude").between(-180, 180)), "Invalid representative coordinates.")
pairs = stations.crossJoin(F.broadcast(representatives))
a = (F.pow(F.sin(F.radians(F.col("request_latitude") - F.col("station_latitude")) / 2), 2)
     + F.cos(F.radians("station_latitude")) * F.cos(F.radians("request_latitude"))
     * F.pow(F.sin(F.radians(F.col("request_longitude") - F.col("station_longitude")) / 2), 2))
pairs = pairs.withColumn("weather_mapping_distance_km",
    6371.0088 * 2 * F.asin(F.sqrt(F.least(F.lit(1.0), F.greatest(F.lit(0.0), a)))))
rank = Window.partitionBy("station_id").orderBy("weather_mapping_distance_km", "weather_request_class")
mapping = (pairs.withColumn("_rank", F.row_number().over(rank)).filter(F.col("_rank") == 1).drop("_rank")
    .withColumn("weather_mapping_method", F.lit("nearest_archived_request_coordinate"))
    .withColumn("station_geography_policy", F.lit("static_retrospective_backfill"))
    .withColumn("station_snapshot_cutoff_utc", F.lit(station_cutoff))
    .withColumn("input_package_sha256", F.lit(package_sha256)))
assert_unique(mapping, ["station_id"], "station mappings")
assert mapping.count() == station_count
print("Most distant station assignments:")
display(mapping.orderBy(F.desc("weather_mapping_distance_km")).limit(15))
assert_empty(mapping.filter(F.col("weather_mapping_distance_km") > MAX_MAPPING_DISTANCE_KM),
             "Stations exceed the configured mapping distance. Review geographic coverage.")
print("All stations mapped within the configured distance.")
print("Expected station forecast rows:", summary.origins * len(HORIZONS) * station_count)


Most distant station assignments:


station_id,station_metadata_fetched_at_utc,station_name,station_latitude,station_longitude,station_location_version_count,weather_request_class,representative_location_id,request_latitude,request_longitude,weather_mapping_distance_km,weather_mapping_method,station_geography_policy,station_snapshot_cutoff_utc,input_package_sha256
1843486301383097150,2026-10-18T02:42:08.995Z,41 Ave & 80 St,40.74586,-73.88539,1,21,4,40.7237521,-73.9769683,8.098223033347221,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1836581441921155232,2026-10-18T02:42:08.995Z,79 St & 37 Ave,40.74937,-73.88714,1,21,4,40.7237521,-73.9769683,8.086761782418918,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1843486390605672640,2026-10-18T02:42:08.995Z,Ithaca St & Baxter Ave,40.74638,-73.88322,1,22,9,40.7510343,-73.7879488,8.042232807738845,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1838617691511229014,2026-10-18T02:42:08.995Z,81 St & 35 Ave,40.75179,-73.88568,1,25,41,40.8043339,-73.951292,8.041024778616311,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1843486189591985058,2026-10-18T02:42:08.995Z,79 St & Roosevelt Ave,40.74718,-73.88675,1,21,4,40.7237521,-73.9769683,8.0354232583718,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1906248765431831024,2026-10-18T02:42:08.995Z,82 St & Broadway,40.74281,-73.88296,1,11,17,40.691507,-73.9499048,8.02344159595579,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
2009671276352543716,2026-10-18T02:42:08.995Z,34 Ave & 83 St,40.753935004830225,-73.88390148299196,1,25,41,40.8043339,-73.951292,7.975208217162462,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1828967830172187422,2026-10-18T02:42:08.995Z,84 St & Roosevelt Ave,40.74797,-73.88204,1,22,9,40.7510343,-73.7879488,7.933386926146123,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1869754328201607270,2026-10-18T02:42:08.995Z,79 St & Woodside Ave,40.74364,-73.88622,1,11,17,40.691507,-73.9499048,7.900123010316051,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1828968327357059242,2026-10-18T02:42:08.995Z,85 St & Northern Blvd,40.75601,-73.88266,1,25,41,40.8043339,-73.951292,7.890964243892821,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9


All stations mapped within the configured distance.
Expected station forecast rows: 250151814


## 10. Save the compact dataset

We save the weather-class table and station mapping with a shared build identifier, then create the station forecast view. Rerunning this cell with the same `DATASET_LABEL` overwrites both component tables and replaces the view.


In [0]:
# Persisting the reusable components and a reproducible dataset identifier.
build_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
weather_saved = (weather.withColumn("input_package_sha256", F.lit(package_sha256))
    .withColumn("weather_model", F.lit("ecmwf_ifs"))
    .withColumn("assumed_availability_delay_hours", F.lit(ASSUMED_AVAILABILITY_DELAY_HOURS))
    .withColumn("dataset_build_id", F.lit(build_id)))
mapping_saved = mapping.withColumn("dataset_build_id", F.lit(build_id))
(weather_saved.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .partitionBy("origin_year", "origin_month").saveAsTable(CLASS_TABLE))
mapping_saved.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(MAPPING_TABLE)

# Selecting weather columns once while avoiding duplicate mapping/provenance names.
mapping_columns = ["station_id", "station_name", "station_latitude", "station_longitude",
    "station_metadata_fetched_at_utc", "station_location_version_count", "weather_mapping_distance_km",
    "weather_mapping_method", "station_geography_policy", "station_snapshot_cutoff_utc"]
map_sql = ", ".join(f"m.`{name}`" for name in mapping_columns)
spark.sql(f"""CREATE OR REPLACE VIEW {FORECAST_VIEW} AS
SELECT {map_sql}, w.*
FROM {MAPPING_TABLE} m
JOIN {CLASS_TABLE} w
 ON m.weather_request_class = w.weather_request_class
 AND m.dataset_build_id = w.dataset_build_id
""")
print("Saved weather:", CLASS_TABLE)
print("Saved mapping:", MAPPING_TABLE)
print("Station forecast view:", FORECAST_VIEW)
print("Build ID:", build_id)
display(spark.table(FORECAST_VIEW).limit(6))


Saved weather: workspace.citibike_weather.archive_v1_weather_classes
Saved mapping: workspace.citibike_weather.archive_v1_station_weather_mapping
Station forecast view: workspace.citibike_weather.archive_v1_station_forecast
Build ID: 20261018T033852681910Z


station_id,station_name,station_latitude,station_longitude,station_metadata_fetched_at_utc,station_location_version_count,weather_mapping_distance_km,weather_mapping_method,station_geography_policy,station_snapshot_cutoff_utc,forecast_origin_utc,target_hour_utc,selected_model_run_utc,selected_run_available_utc,valid_time_utc,forecast_temperature_f,forecast_precipitation_in,forecast_rain_in,forecast_snowfall_in,forecast_wind_speed_mph,weather_request_class,forecast_horizon,representative_location_id,request_latitude,request_longitude,model_grid_latitude,model_grid_longitude,selected_run_age_hours,weather_lead_hours,is_fallback_weather_run,selected_forecast_hours,selected_weather_source,selected_raw_json_relative_path,origin_year,origin_month,weather_forecast_has_missing,input_package_sha256,weather_model,assumed_availability_delay_hours,dataset_build_id
f06c01c3-fd7f-443c-a0c7-fc9233d77f6a,67 St & Erik Pl,40.633385,-74.016562,2026-10-18T02:42:08.995Z,1,1.472986843016889,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.1,0.0,0.0,0.0,7.7,4,1,14,40.62483367348954,-74.0298925091807,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
fd0e86e3-079d-40aa-8231-ec2cbdbd9e7f,34 St & 4 Ave,40.655278,-74.003101,2026-10-18T02:42:08.995Z,1,0.7621916192983097,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.8,0.0,0.0,0.0,8.2,9,1,228,40.652354155320815,-74.0112729025337,40.597538,-73.96039,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
ffaa8f6a-a4ea-4d3a-a57c-f42f77cd4581,Thomas S. Boyland St & Fulton St,40.67854,-73.91367,2026-10-18T02:42:08.995Z,1,3.3784457643042303,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,66.6,0.0,0.0,0.0,6.7,11,1,17,40.69150702163218,-73.94990480442759,40.667835,-73.93768,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
e25f8dee-c110-489f-a978-464ac5c84ca2,Nevins St & Schermerhorn St,40.68737247650719,-73.98176059126854,2026-10-18T02:42:08.995Z,1,2.659674352854638,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,66.5,0.0,0.0,0.0,6.7,12,1,195,40.675548647131166,-74.00917842775038,40.667835,-73.93768,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
2206782596968628766,78 Ave & Queens Blvd S,40.7149,-73.83247,2026-10-18T02:42:08.995Z,1,5.311079210449747,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000

## 11. Validate the saved join

We reconcile the component counts and the implied station-level row count. For one forecast origin, we also check that each station has exactly one row per horizon.


In [0]:
# Reconciling persisted component counts and testing one complete origin slice.
saved_weather = spark.table(CLASS_TABLE)
saved_mapping = spark.table(MAPPING_TABLE)
assert saved_weather.count() == summary.rows
assert saved_mapping.count() == station_count
assert_unique(saved_mapping, ["station_id"], "saved station mappings")
weights = saved_mapping.groupBy("weather_request_class").agg(F.count("*").alias("stations_in_class"))
weighted_rows = saved_weather.join(weights, "weather_request_class").agg(F.sum("stations_in_class")).first()[0]
expected_rows = summary.origins * len(HORIZONS) * station_count
assert weighted_rows == expected_rows
sample = spark.table(FORECAST_VIEW).filter(F.col("forecast_origin_utc") == F.lit(summary.first_origin))
assert sample.count() == station_count * len(HORIZONS)
assert_unique(sample, ["station_id", "forecast_origin_utc", "forecast_horizon"], "sample station forecast keys")
print("Saved class rows:", summary.rows)
print("Saved station mappings:", station_count)
print("Complete station-level row count implied by the validated join:", expected_rows)
print("Sample origin join passed:", summary.first_origin)


Saved class rows: 2780568
Saved station mappings: 2519
Complete station-level row count implied by the validated join: 250151814
Sample origin join passed: 2024-08-10 07:00:00


## 12. Inspect the final dataset

We inspect the view's grain, columns, schema, and sample forecasts before using it in the modeling pipeline.


In [0]:
# Inspecting the structure of the final station forecast dataset.
final = spark.table(FORECAST_VIEW)
columns = final.columns
key_columns = ["station_id", "forecast_origin_utc", "forecast_horizon", "target_hour_utc"]

# Using counts already validated above instead of counting the full view again.
print("Dataset:", FORECAST_VIEW)
print("Grain: one row per station_id × forecast_origin_utc × forecast_horizon")
print("Stations:", station_count)
print("Forecast origins:", summary.origins, f"({summary.first_origin} to {summary.last_origin} UTC)")
print("Horizons:", HORIZONS)
print("Rows:", expected_rows)
print("Columns:", len(columns))

# Grouping columns by role to show how the dataset is organized.
roles = {
    "Keys": key_columns,
    "Station": [c for c in columns if c.startswith("station_") and c not in key_columns],
    "Weather values": WEATHER_VALUES + ["weather_forecast_has_missing"],
    "Mapping": [c for c in columns if c.startswith("weather_mapping_")],
}
assigned = {c for group in roles.values() for c in group}
roles["Run and provenance"] = [c for c in columns if c not in assigned]
display(spark.createDataFrame(
    [(role, len(names), ", ".join(names)) for role, names in roles.items()],
    ["role", "column_count", "columns"],
))

# Showing column names and data types.
final.printSchema()

# Showing all horizons for one station at the first origin.
example_station = final.select("station_id").orderBy("station_id").first()[0]
display(
    final.filter((F.col("station_id") == example_station)
                 & (F.col("forecast_origin_utc") == F.lit(summary.first_origin)))
    .select(*key_columns, *WEATHER_VALUES, "weather_mapping_distance_km", "is_fallback_weather_run")
    .orderBy("forecast_horizon")
)

## 13. Optional: materialize station forecasts

We keep `MATERIALIZE_STATION_TABLE = False` unless a physical station-level table is needed. When enabled, the expanded data is written one month at a time. An interrupted run can resume from this cell while the build identifier remains unchanged.


In [0]:
# Materializing a physical station table only when requested.
if MATERIALIZE_STATION_TABLE:
    if spark.catalog.tableExists(EXPANDED_TABLE):
        old_builds = [r[0] for r in spark.table(EXPANDED_TABLE).select("dataset_build_id").distinct().collect()]
        assert not old_builds or old_builds == [build_id], "Use a new DATASET_LABEL for a different build."
    months = saved_weather.select("origin_year", "origin_month").distinct().orderBy("origin_year", "origin_month").collect()
    for row in months:
        year, month = row.origin_year, row.origin_month
        predicate = f"origin_year = {year} AND origin_month = {month}"
        month_data = spark.table(FORECAST_VIEW).filter(predicate)
        writer = month_data.write.format("delta").mode("overwrite")
        if spark.catalog.tableExists(EXPANDED_TABLE):
            writer.option("replaceWhere", predicate).saveAsTable(EXPANDED_TABLE)
        else:
            writer.partitionBy("origin_year", "origin_month").saveAsTable(EXPANDED_TABLE)
        print(f"Saved station forecasts: {year}-{month:02d}")
    assert spark.table(EXPANDED_TABLE).count() == expected_rows
    print("Completed materialized dataset:", EXPANDED_TABLE)
else:
    print("Using the complete persistent view:", FORECAST_VIEW)


## 14. Export the dataset

We export the weather-class table and station mapping to Parquet with a transfer manifest. Together, these compact components recreate the station forecast view in another workspace. We share the export only after the cell completes and writes `transfer_manifest.json`.


In [0]:
# Exporting the compact completed dataset; optionally exporting expanded monthly files too.
if EXPORT_PORTABLE_DATA:
    transfer_root = f"{EXPORT_ROOT}/{build_id}"
    export_months = saved_weather.select("origin_year", "origin_month").distinct().orderBy("origin_year", "origin_month").collect()
    exported_months = []
    for row in export_months:
        year, month = row.origin_year, row.origin_month
        part = f"year={year}/month={month:02d}"
        predicate = f"origin_year = {year} AND origin_month = {month}"
        (saved_weather.filter(predicate).repartition(EXPORT_PARTITIONS_PER_MONTH)
         .write.mode("overwrite").parquet(f"{transfer_root}/weather_classes/{part}"))
        if EXPORT_EXPANDED_DATA:
            (spark.table(FORECAST_VIEW).filter(predicate).repartition(EXPORT_PARTITIONS_PER_MONTH)
             .write.mode("overwrite").parquet(f"{transfer_root}/station_forecasts/{part}"))
        exported_months.append({"origin_year": year, "origin_month": month})
        print("Exported:", part)
    saved_mapping.write.mode("overwrite").parquet(f"{transfer_root}/station_mapping")
    transfer_manifest = {
        "format_version": 1, "dataset_build_id": build_id,
        "input_package_sha256": package_sha256, "weather_model": "ecmwf_ifs",
        "class_rows": summary.rows, "station_count": station_count,
        "station_forecast_rows": expected_rows,
        "first_origin_utc": summary.first_origin.isoformat(),
        "last_origin_utc": summary.last_origin.isoformat(),
        "station_snapshot_cutoff_utc": station_cutoff.isoformat(),
        "horizons": HORIZONS, "months": exported_months,
        "mapping_method": "nearest_archived_request_coordinate",
        "station_geography_policy": "static_retrospective_backfill",
        "assumed_availability_delay_hours": ASSUMED_AVAILABILITY_DELAY_HOURS,
        "max_mapping_distance_km": MAX_MAPPING_DISTANCE_KM,
        "expanded_export_included": EXPORT_EXPANDED_DATA,
        "units": {"temperature": "F", "precipitation": "inch", "rain": "inch",
                  "snowfall": "inch", "wind_speed": "mph"},
    }
    Path(f"{transfer_root}/transfer_manifest.json").write_text(json.dumps(transfer_manifest, indent=2))
    print("Complete portable dataset:", transfer_root)
    print("Download this build directory from Catalog Explorer, then transfer its ZIP.")
else:
    print("Portable export disabled.")


Exported: year=2024/month=08
Exported: year=2024/month=09
Exported: year=2024/month=10
Exported: year=2024/month=11
Exported: year=2024/month=12
Exported: year=2025/month=01
Exported: year=2025/month=02
Exported: year=2025/month=03
Exported: year=2025/month=04
Exported: year=2025/month=05
Exported: year=2025/month=06
Exported: year=2025/month=07
Exported: year=2025/month=08
Exported: year=2025/month=09
Exported: year=2025/month=10
Exported: year=2025/month=11
Exported: year=2025/month=12
Exported: year=2026/month=01
Exported: year=2026/month=02
Exported: year=2026/month=03
Exported: year=2026/month=04
Exported: year=2026/month=05
Exported: year=2026/month=06
Complete portable dataset: /Volumes/workspace/citibike_weather/transfer/exports/archive_v1/20261018T033852681910Z
Download this build directory from Catalog Explorer, then transfer its ZIP.


## 15. Import into another workspace

We upload and extract the exported ZIP into a Unity Catalog volume. `RECIPIENT_INPUT_ROOT` must point directly to the folder containing `weather_classes`, `station_mapping`, and `transfer_manifest.json`.

We copy the following standalone cell into the destination notebook, set the input path and destination namespace, and enable `RUN_RECIPIENT_IMPORT`. The original weather project and source station table are not required. We choose a new namespace if the destination tables already exist.


In [0]:
# Importing independently so this cell also works alone in the recipient workspace.
from pathlib import Path
import json
import re
from pyspark.sql import functions as F

RUN_RECIPIENT_IMPORT = False
RECIPIENT_INPUT_ROOT = "/Volumes/workspace/citibike_weather/transfer/received/extracted_build"
RECIPIENT_NAMESPACE = "workspace.citibike_weather_received"

# Reconstructing managed tables from the transferred Parquet components.
if RUN_RECIPIENT_IMPORT:
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    def assert_empty(frame, message):
        assert not frame.limit(1).count(), message
    def assert_unique(frame, keys, label):
        assert_empty(frame.groupBy(*keys).count().filter(F.col("count") != 1), label)
    received_mapping_columns = ["station_id", "station_name", "station_latitude", "station_longitude",
        "station_metadata_fetched_at_utc", "station_location_version_count", "weather_mapping_distance_km",
        "weather_mapping_method", "station_geography_policy", "station_snapshot_cutoff_utc"]
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", RECIPIENT_NAMESPACE)
    incoming = json.loads(Path(f"{RECIPIENT_INPUT_ROOT}/transfer_manifest.json").read_text())
    assert incoming["format_version"] == 1
    incoming_weather = (spark.read.option("recursiveFileLookup", "true")
                        .parquet(f"{RECIPIENT_INPUT_ROOT}/weather_classes"))
    incoming_mapping = spark.read.parquet(f"{RECIPIENT_INPUT_ROOT}/station_mapping")
    assert incoming_weather.count() == incoming["class_rows"]
    assert incoming_mapping.count() == incoming["station_count"]
    assert_unique(incoming_weather, ["forecast_origin_utc", "forecast_horizon", "weather_request_class"], "received class keys")
    assert_unique(incoming_mapping, ["station_id"], "received mappings")
    for frame in [incoming_weather, incoming_mapping]:
        assert_empty(frame.filter(F.col("dataset_build_id").isNull()
            | (F.col("dataset_build_id") != incoming["dataset_build_id"])), "Mixed export builds.")
    assert_empty(incoming_mapping.select("weather_request_class").distinct().join(
        incoming_weather.select("weather_request_class").distinct(), "weather_request_class", "left_anti"),
        "A received station has no weather class.")
    incoming_weights = incoming_mapping.groupBy("weather_request_class").agg(F.count("*").alias("n"))
    assert incoming_weather.join(incoming_weights, "weather_request_class").agg(F.sum("n")).first()[0] == incoming["station_forecast_rows"]
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RECIPIENT_NAMESPACE}")
        # Checking both destination names before creating either table.
    for table_name in ["weather_classes", "station_weather_mapping"]:
        destination = f"{RECIPIENT_NAMESPACE}.{table_name}"
        assert not spark.catalog.tableExists(destination), (
            f"Destination already exists: {destination}. "
            "Choose a different RECIPIENT_NAMESPACE."
        )

    # Making the validated export components available to SQL.
    incoming_weather.createOrReplaceTempView("received_weather_to_import")
    incoming_mapping.createOrReplaceTempView("received_mapping_to_import")

    # Creating managed Delta tables using the tested SQL approach.
    spark.sql(f"""
        CREATE TABLE {RECIPIENT_NAMESPACE}.weather_classes
        USING DELTA
        PARTITIONED BY (origin_year, origin_month)
        AS SELECT * FROM received_weather_to_import
    """)

    spark.sql(f"""
        CREATE TABLE {RECIPIENT_NAMESPACE}.station_weather_mapping
        USING DELTA
        AS SELECT * FROM received_mapping_to_import
    """)
    received_map_sql = ", ".join(f"m.`{name}`" for name in received_mapping_columns)
    spark.sql(f"""CREATE VIEW {RECIPIENT_NAMESPACE}.station_forecast AS
    SELECT {received_map_sql}, w.*
    FROM {RECIPIENT_NAMESPACE}.station_weather_mapping m
    JOIN {RECIPIENT_NAMESPACE}.weather_classes w
    ON m.weather_request_class = w.weather_request_class
    AND m.dataset_build_id = w.dataset_build_id""")
    print("Received dataset:", f"{RECIPIENT_NAMESPACE}.station_forecast")
else:
    print("Recipient import is disabled in the source workspace.")


## Before modeling

- We reconcile GBFS station UUIDs with trip station IDs using dated mappings where available.
- We verify the actual forecast date range against the agreed training, validation, and test periods.
- We retain mapping distances, fallback metadata, and missingness flags.
- We keep forecast origin, target hour, and horizon separate, and check predictor availability at each origin.
- We apply chronological splits and fit learned preprocessing on training data only.

## References

- [Databricks Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations)
- [Working with Unity Catalog volume files](https://docs.databricks.com/aws/en/volumes/volume-files)
- [Unity Catalog volumes](https://docs.databricks.com/aws/en/volumes)
- [Open-Meteo Single Runs API](https://open-meteo.com/en/docs/single-runs-api)
